In [ ]:
import os
import re
import warnings
import requests
import time
import pandas as pd
from tqdm import tqdm
from openpyxl import load_workbook
from bs4 import BeautifulSoup

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

In [ ]:
folder_path = './cons/datas/XBRL_2026_Q2/'
folder_transit_path = './cons/transit/'
folder_output_path = './cons/output/'

tahun = 2026
kuartal = 'Q2'

In [ ]:
def get_file_list(directory_path):

    files = []
    for entry in os.listdir(directory_path):
        full_path = os.path.join(directory_path, entry)
        if os.path.isfile(full_path):
            files.append(entry)
    return files

In [ ]:
def extract_file_info(file_list):

    pattern = r'FinancialStatement-(\d{4})-([IVX]+)-([A-Z]+)\.xlsx'

    records = []  # list of dict, tiap dict jadi 1 row di DataFrame nantinya

    for filename in file_list:
        match = re.match(pattern, filename)

        if match:
            records.append({
                "filename": filename,
                "year": match.group(1),
                "quarter": match.group(2),
                "company_code": match.group(3)
            })
        else:
            records.append({
                "filename": filename,
                "year": None,
                "quarter": None,
                "company_code": None
            })

    df = pd.DataFrame(records)

    return df

In [ ]:
def get_name_sheets(file_list, directory_path, keyword):

    sheet_list = {
        "kode perusahaan": [],
        "nama file": [],
        "nama sheet": []
    }

    for data_file in file_list.itertuples(index=False):
        try:
            wb = load_workbook(f"{directory_path}/{data_file.filename}", read_only=True)
            for ws in wb.worksheets:
                for row in ws.iter_rows(values_only=True):
                    if keyword in row:
                        sheet_list["kode perusahaan"].append(data_file.company_code)
                        sheet_list["nama file"].append(data_file.filename)
                        sheet_list["nama sheet"].append(ws.title)
                        break
            wb.close()
        except Exception as e:
            print(f"Gagal membuka file {data_file.filename}: {e}")

    print(f"Jumlah sheet yang mengandung kata kunci '{keyword}': {len(sheet_list['nama sheet'])}")
    print(f"{keyword} selesai")

    return pd.DataFrame(sheet_list)
 


In [ ]:
list_name_files=get_file_list('./cons/datas/XBRL_2026_Q2/')
list_info_files=extract_file_info(list_name_files)
list_name_sheets_GI=get_name_sheets(list_info_files, './cons/datas/XBRL_2026_Q2', 'General information')
list_name_sheets_PK=get_name_sheets(list_info_files, './cons/datas/XBRL_2026_Q2', 'Statement of financial position')
list_name_sheets_AK=get_name_sheets(list_info_files, './cons/datas/XBRL_2026_Q2', 'Statement of cash flows')
list_name_sheets_LR=get_name_sheets(list_info_files, './cons/datas/XBRL_2026_Q2', 'Statement of profit or loss and other comprehensive income')

In [ ]:
def xbrl_scraper(data):
    print("Memulai pengambilan data:")

    unique_sheets = data['nama sheet'].unique()  # ambil daftar sheet unik sekali saja

    for target_sheet in tqdm(unique_sheets):
        data_filtered = data[data['nama sheet'] == target_sheet].reset_index(drop=True)
        wadah_transit = pd.DataFrame()

        for _, row in data_filtered.iterrows():
            try:
                current_sheet = str(row['nama sheet'])  # <- variabel BARU, tidak menimpa target_sheet

                if not current_sheet:
                    print(f"Skipping {row['nama file']} karena sheet {row['nama sheet']} tidak ditemukan.")
                    continue

                file_path = f"{folder_path}/{row['nama file']}"
                file_target = pd.read_excel(file_path, sheet_name=current_sheet, index_col=None)
                file_target = file_target.dropna(how="all").T
                file_target = file_target.drop(file_target.columns[0], axis=1).reset_index(drop=True).drop(3)
                file_target.loc[0, 2] = "Tanggal"
                file_target.columns = file_target.iloc[0].str.lower()
                file_target = file_target[1:].dropna(axis=1, how="all")

                # broadcast otomatis oleh pandas, tidak hardcode [x] * 2
                file_target["kode perusahaan"] = row["kode perusahaan"]

                wadah_transit = pd.concat([wadah_transit, file_target], ignore_index=True)

            except Exception as e:
                print(f"Error pada {row['kode perusahaan']}: {e}")

        wadah_transit.to_excel(f"{folder_transit_path}{target_sheet}.xlsx", index=False)

    print("Pengumpulan data selesai.\r")

In [ ]:
xbrl_scraper(list_name_sheets_AK)
xbrl_scraper(list_name_sheets_LR)
xbrl_scraper(list_name_sheets_PK)

In [ ]:
def merge_with_coalesce(left, right, on):
    kolom_overlap = [
        col for col in left.columns
        if col in right.columns and col not in on
    ]

    merged = pd.merge(
        left, right,
        on=on,
        how="outer",
        suffixes=("", "_dup")  # tetap perlu, sebagai penampung sementara
    )

    for col in kolom_overlap:
        dup_col = f"{col}_dup"
        if dup_col in merged.columns:
            # combine_first: ambil nilai dari kolom asli (left),
            # kalau NaN, isi dari kolom "_dup" (right)
            merged[col] = merged[col].combine_first(merged[dup_col])
            merged.drop(columns=[dup_col], inplace=True)  # hapus segera, jangan menumpuk

    return merged

In [ ]:
def gabungkan_data(jenis_laporan):
    daftar_df = []

    for sheet in jenis_laporan:
        df_sheet = pd.read_excel(f"{folder_transit_path}{sheet}.xlsx")
        df_sheet.columns = [str(col).strip().lower() for col in df_sheet.columns]

        if "kode perusahaan" not in df_sheet.columns or "tanggal" not in df_sheet.columns:
            raise KeyError(
                f"Sheet '{sheet}' tidak memiliki kolom 'kode perusahaan' dan/atau 'tanggal', "
                f"kolom tersedia: {list(df_sheet.columns)}"
            )

        daftar_df.append(df_sheet)

    gabung_all = daftar_df[0]
    for df_next in daftar_df[1:]:
        gabung_all = merge_with_coalesce(gabung_all, df_next, on=["kode perusahaan", "tanggal"]) # <--- merged with coalesce disini

    kolom_awal = ["kode perusahaan", "tanggal"]
    gabung_all = gabung_all[kolom_awal + [col for col in gabung_all.columns if col not in kolom_awal]]
    gabung_all = gabung_all.fillna(0)

    return gabung_all

In [ ]:
def general_information(data):
    info_entitas = pd.concat([
        pd.read_excel(f"{folder_path}/{file}", sheet_name="1000000").T.dropna(how="all").reset_index(drop=True).drop(2)
        for file in data["nama file"]
    ], ignore_index=True)
    
    info_entitas = info_entitas.drop_duplicates().reset_index(drop=True)
    info_entitas.columns = info_entitas.iloc[0].str.lower()
    info_entitas = info_entitas.drop(info_entitas.columns[:2], axis=1).drop(0)
    
    return info_entitas

In [ ]:
data_AK = gabungkan_data(list_name_sheets_AK["nama sheet"].unique())
data_LR = gabungkan_data(list_name_sheets_LR["nama sheet"].unique())
data_PK = gabungkan_data(list_name_sheets_PK["nama sheet"].unique())
data_GI = general_information(list_name_sheets_GI["nama sheet"].unique())

In [ ]:
def pemisah_data(df):
    current_q = df.iloc[::2].reset_index(drop=True)
    previous_q = df.iloc[1::2].reset_index(drop=True)
    return current_q, previous_q

In [ ]:
PK_currentQ, PK_previousQ = pemisah_data(data_AK)
LR_currentQ, LR_previousQ = pemisah_data(data_LR)
RB_currentQ, RB_previousQ = pemisah_data(data_PK)

In [ ]:
def stock_latest_googlefinance (data) :

    datas = data["kode perusahaan"].unique()

    harga_stock = []

    for ticker in tqdm(datas) :

        url = f'https://www.google.com/finance/quote/{ticker}:IDX?hl=en'
        response = requests.get(url)
        soup = BeautifulSoup(response.content, 'html.parser')

        stocknya = soup.find('div', class_='AHmHk').text
        stocknya = "".join(filter(str.isdigit, stocknya))
        stocknya = int(stocknya) // 100
        
        harga_stock.append(stocknya)
        time.sleep(1)
        
    all_stocks = {
        "kode perusahaan" : datas,
        "penutupan" : harga_stock
    }

    all_stocks = pd.DataFrame(all_stocks)
    
    print("Selesai")
    return all_stocks

In [ ]:
with pd.ExcelWriter(f"{folder_output_path}Data_Laporan_{tahun}_{kuartal}.xlsx", engine='openpyxl') as writer:
    general_info.to_excel(writer, sheet_name="gen_info", index=False)
    PK_currentQ.to_excel(writer, sheet_name="pk_now", index=False)
    LR_currentQ.to_excel(writer, sheet_name="lr_now", index=False)
    RB_currentQ.to_excel(writer, sheet_name="rb_now", index=False)
    PK_previousQ.to_excel(writer, sheet_name="pk_prev", index=False)
    LR_previousQ.to_excel(writer, sheet_name="lr_prev", index=False)
    RB_previousQ.to_excel(writer, sheet_name="rb_prev", index=False)